# Create a Knowledge Graph from Text

## Task 1: Import Libraries

In [1]:
import wikipedia as wp 
import re
import requests
import spacy
import spacy_transformers
from spacy import displacy
from spacy.matcher import Matcher
import networkx as nx
from pyvis.network import Network

## Task 2: Load the Data

In [2]:
# Set the language of the response
wp.set_lang("en")

# Obtain and store the data
title = "New York"
data = wp.page(title).content

# View the data
print(data)

New York, often called New York City (NYC), is the most populous city in the United States. It is located at the southern tip of New York State on New York Harbor, one of the world's largest natural harbors. The city comprises five boroughs, each coextensive with its respective county. It is the geographical and demographic center of both the Northeast megalopolis and the New York metropolitan area, the largest metropolitan area in the United States by both population and urban area. New York is a global center of finance and commerce, culture, technology, entertainment and media, academics and scientific output, the arts and fashion, and, as home to the headquarters of the United Nations, international diplomacy.
With an estimated population of 8,584,629 in July 2025, distributed over 300.46 square miles (778.2 km2), New York is the most densely populated major city in the United States. New York City has more than double the population of Los Angeles, the country's second-most populo

## Task 3: Preprocess the Data

In [3]:
# Convert the data to lowercase and replace new lines
data = data.lower().replace('\n', "")

# Remove the last part of the text, certain punctuation marks, headings, as well as any text within the parentheses
data = re.sub('== see also ==.*|[@#:&\\"]|===.*?===|==.*?==|\\(.*?\\)', '', data)

# View the data
print(data)

new york, often called new york city , is the most populous city in the united states. it is located at the southern tip of new york state on new york harbor, one of the world's largest natural harbors. the city comprises five boroughs, each coextensive with its respective county. it is the geographical and demographic center of both the northeast megalopolis and the new york metropolitan area, the largest metropolitan area in the united states by both population and urban area. new york is a global center of finance and commerce, culture, technology, entertainment and media, academics and scientific output, the arts and fashion, and, as home to the headquarters of the united nations, international diplomacy.with an estimated population of 8,584,629 in july 2025, distributed over 300.46 square miles , new york is the most densely populated major city in the united states. new york city has more than double the population of los angeles, the country's second-most populous city. over 20.

## Task 4: Recognize Named Entities

In [4]:
# Load a language model
nlp = spacy.load('en_core_web_lg')
doc=nlp(data)

# Display the entities in the doc
displacy.render(doc,style="ent",jupyter=True)

## Task 5: Compute Coreference Clusters

In [5]:
# Add the coreference resolution component in the pipeline
nlp.add_pipe('coreferee')

# Pass the data to the language model 
doc = nlp(data)

# Print resolved coreferences, if any
doc._.coref_chains.print()

0: york(1), city(13), it(19), york(72), york(92), york(150), city(157)
1: harbor(33), city(45), its(53), it(57)
2: states(83), states(161)
3: city(165), city(193), its(202), city(219), city(231), its(233), city(274)
4: angeles(174), country(177)
5: world(224), world(269), world(292)
6: states(247), city(266)
7: million(296), its(303)
8: amsterdam(307), amsterdam(327)
9: city(335), city(340)
10: ii(356), his(361)
11: britain(385), city(388)
12: states(396), city(404), its(412)
13: manhattan(415), manhattan(431)
14: city(438), city(456)
15: world(444), world(459), world(477), world(517)
16: york(467), york(491), york(513), city(525)
17: countries(502), their(540)
18: world(530), world(576), world(604), world(633)
19: city(553), city(568), city(573), city(608)
20: york(641), york(650), city(674), york(695)
21: james(652), him(662)
22: netherland(670), netherland(688)
23: england(680), it(682)
24: algonquians(715), their(721)
25: harbor(761), harbor(807), harbor(838)
26: verrazzano(772), h

## Task 6: Resolve Coreferences

In [6]:
resolved_data = ""
for token in doc:
    resolved_coref = doc._.coref_chains.resolve(token)
    if resolved_coref:
        resolved_data += " " + " and ".join(r.text for r in resolved_coref)
    elif token.dep_ == "punct":
        resolved_data += token.text
    else:
        resolved_data += " " + token.text
print(resolved_data)

 new york, often called new york city, is the most populous york in the united states. york is located at the southern tip of new york state on new york harbor, one of the world 's largest natural harbors. the harbor comprises five boroughs, each coextensive with harbor respective county. harbor is the geographical and demographic center of both the northeast megalopolis and the new york metropolitan area, the largest metropolitan area in the united states by both population and urban area. new york is a global center of finance and commerce, culture, technology, entertainment and media, academics and scientific output, the arts and fashion, and, as home to the headquarters of the united nations, international diplomacy.with an estimated population of 8,584,629 in july 2025, distributed over 300.46 square miles, new york is the most densely populated major york in the united states. new york city has more than double the population of los angeles, the angeles 's second- most populous c

## Task 7: Extract Relationships

In [7]:
def extract_relationship(sentence):
    doc = nlp(sentence)
    
    first, last = None, None
    
    for chunk in doc.noun_chunks:
        if not first:
            first = chunk
        else:
            last = chunk

    if first and last:
        return (first.text.strip(), last.text.strip(), str(doc[first.end:last.start]).strip())
    
    return (None, None, None)

## Task 8: Create a Graph

In [8]:
# A helper function that prints 5 words per row. Can be used for better readability of a given text.
print_five_words = lambda sentence: '\n'.join(' '.join(sentence.split()[i:i+5]) for i in range(0, len(sentence.split()), 5))

In [9]:
# Create a Network object
graph_doc = nlp(resolved_data)

# Create an empty graph
nx_graph = nx.DiGraph()

for sent in enumerate(graph_doc.sents) :
    if len(sent[1]) > 3:
        (a, b, c) = extract_relationship(str(sent[1]))

        # Add nodes and edges to graph
        if a and b:
            nx_graph.add_node(a, size = 5)
            nx_graph.add_node(b, size = 5)
            nx_graph.add_edge(a, b, weight=1, title=print_five_words(c), arrows="to")

g = Network(notebook=True, cdn_resources='in_line')
g.from_nx(nx_graph)
g.save_graph("/usercode/example.html")

In [12]:
# Run this cell to view the resulting graph i.e. the /usercode/example.html file
from IPython.display import HTML, display
import base64

# Read and encode the HTML as base64
with open("/usercode/example.html", "r", encoding="utf-8") as f:
    html_str = f.read()
    b64_html = base64.b64encode(html_str.encode("utf-8")).decode("utf-8")

# Create an iframe using a data URI
iframe = f"""
<iframe src="data:text/html;base64,{b64_html}" width="100%" height="600" style="border:none;"></iframe>
"""

display(HTML(iframe))


## Task 9: List the Related Entities

In [11]:
print(nx_graph.edges(['manhattan']))

[('manhattan', 'many important universities'), ('manhattan', "the nation 's 360 largest counties"), ('manhattan', 'the second department')]
